# Kriging Sync

Authenticate and load geoscience objects with the same async widget and loader APIs used
by the other Kriging examples. Then run Kriging and load its results synchronously,
without `await` or constructing a compute client.

`Kriging.run(manager, parameters)` blocks until the task finishes. Its result loaders
also block until the requested data is available. The handle receives the manager
explicitly and carries no manager or input state.

## Authentication

Authenticate with `ServiceManagerWidget`, as in the other Kriging examples. Replace
`your-client-id` with your registered application ID, then select the organization,
hub, and workspace in the widget before loading inputs.

Authentication stays async. The widget holds its connection open on the notebook's
event loop until the input-loading cell releases it.

In [ ]:
from evo.notebooks import ServiceManagerWidget

manager = await ServiceManagerWidget.with_auth_code(
    client_id="your-client-id", cache_location="./notebook-data"
).login()

## Load Source PointSet, Target Grid, and Variogram

Load the objects with `object_from_uuid`, as in the other Kriging examples. Replace the
example UUIDs with objects in the workspace selected in the authentication widget.

After loading, close the widget's connection once on the notebook's event loop. This
releases the open HTTP session, not the login or selected context. Blocking compute
and result loaders can then open fresh sessions on their own loop. Run this cell once
after each widget login; to reload inputs, repeat only the three object-loading calls.

In [ ]:
from evo.objects.typed import object_from_uuid

source_pointset = await object_from_uuid(manager, "9100d7dc-44e9-4e61-b427-159635dea22f")
target_grid = await object_from_uuid(manager, "df9c3705-c82e-4f57-af94-b3346b5d58cf")
variogram = await object_from_uuid(manager, "72cd9b83-90f4-4cb0-9691-95728e3f9cbb")

await manager.get_connector().close()

## Import the task

`Kriging` is an immutable task handle. It carries the task identity, not a manager or a
connection. Importing it does not authenticate or contact discovery; each call receives
its context explicitly and reuses that context's private compute client when possible.

In [ ]:
from evo.compute.tasks.geostatistics import Kriging
from evo.compute.tasks.geostatistics.kriging import KrigingParameters

## Task Discovery

The imported name and editor hints come from shipped metadata, but the execution schema
comes from live discovery on the first `Kriging.run(...)` call. Later calls with the
same context reuse its discovery cache until it expires.

For a newly published task without a shipped import name, use
`task("topic", "task-name").run(manager, parameters)` from `evo.compute`.

## Run it

The parameters are the ones the task publishes, and the objects and attributes you already
hold can be passed directly -- the engine turns them into the references the schema asks for.
A missing or misspelled parameter is rejected before anything is submitted.

Prefer the typed attribute (`samples.attributes["..."]`) over a bare name. Both are accepted,
but a name is turned into a JMESPath expression matched against the object, so a name that is
not there selects nothing and the task fails on the service rather than here. An attribute you
hold carries its own key, so there is nothing to guess. Indexing a name the object does *not*
have -- `grid.attributes["kriged_grade"]` -- is how you say "create this one".

In [ ]:
from evo.compute.tasks.common import SearchNeighborhood

search_ellipsoid = variogram.get_ellipsoid().scaled(2.0)
parameters = KrigingParameters(
    source=source_pointset.attributes["Ag_ppm Values"],
    target=target_grid.attributes["kriged_grade"],
    variogram=variogram,
    search=SearchNeighborhood(ellipsoid=search_ellipsoid, max_samples=20),
    block_discretisation={"nx": 3, "ny": 3, "nz": 2},
)

result = Kriging.run(manager, parameters, preview=True)
result

## Read the Results

Generic execution returns a `SyncTaskResult` with blocking output loaders. A bespoke
Kriging runner returns the existing `KrigingResult`, whose own loader API stays async
for compatibility.

The cell below handles both without `await`. For a bespoke result, `SyncResultNode`
wraps its public target reference and loads the object and estimated column through
the existing blocking output API. The original result type is unchanged.

In [ ]:
from evo.compute import SyncResultNode, SyncTaskResult

if isinstance(result, SyncTaskResult):
    estimated = result.target.load()
    dataframe = result.target.attribute.to_dataframe()
else:
    target = SyncResultNode(
        {"reference": result.target_reference},
        {"output": "geoscience-object"},
        manager,
    )
    estimated = target.load()
    dataframe = target.to_dataframe(result.attribute_name)

In [ ]:
estimated

In [ ]:
dataframe.head()

## Async Setup, Synchronous Compute and Results

Widget authentication and input loading use the usual async SDK APIs. The input-loading
cell then releases the widget's open connection before blocking execution begins.

`Kriging.run(...)` and the generic or wrapped output loaders are synchronous. They
wait for completion without moving an open session between event loops. The notebook's
event loop is blocked while these calls run; computation and result downloads run on
the private compute bridge. No public coroutine helper is needed for authentication,
input loading, or result handling.

## If Something Goes Wrong

- **`SelectionError`**: select an organization, hub, and workspace in the authentication
  widget before loading objects or running compute.
- **Object-loading errors**: replace the example UUIDs with objects in the selected workspace.
- **`ParameterValidationError`**: the parameters did not match the task's schema. The
  message names the parameter; nothing was submitted.
- **`SyncBridgeError: this connection was first used on a different event loop`**: an HTTP
  session is still open on the notebook's loop. Complete the input-loading cleanup before
  running synchronous compute, and do not keep a caller-loop connection open around
  blocking calls.